# Prompt lab: segmentation and scoring prompts

Try changes to the prompts before the 100-slide cluster run. Everything lives in `configs/segmentation.json`:

| What | Key | Where it runs |
|---|---|---|
| SAM 3 text prompts (what to segment) | `sam3.prompts`, `sam3.retry_prompts` | workers (SAM 3) |
| SAM 3 confidence / box filters | `sam3.conf`, `sam3.nms_iou`, `sam3.max_box_area_ratio` | workers |
| Mask clean-up (split / fragment / contained masks) | `postprocess.*` | workers |
| **Per-feather VLM score** (the rubric) | `feather_judge.prompt` | VLM server, from here or the workers |
| Missed-feather check on the numbered overlay (review flags only) | `inventory.prompt` | VLM server, from here or the workers |

The old slide-level judge (bird ID, date and one 1-10 overlay score) is retired: the label extractor owns bird ID and date, and the inventory plus the per-feather checks cover the rest.

**Section 2** compares two feather-judge prompts on real crops, from here (VLM server), cached per prompt and crop.
Optionally fill in your own calls per crop (good/bad, and each check) in a CSV: both prompts are then scored against you, overall and check by check.
**Section 3** sends slides through the workers with different SAM 3 prompts/settings (no redeploy), once the workers
run this branch's code; it compares feather counts, clean-up actions, judge scores and overlays.
Kernel: **bird-feather-analysis (pixi)**.

## 1. Parameters and the current prompts

In [ ]:
SEGMENTATION_DIR = "../data/sample_runs/test-sample30-seg"   # crops to judge (any segmentation output folder)
SLIDES = None              # e.g. ["A3818 2002-im1708.jpg", "A0012 1996-im1058.jpg"]; None = all in the folder
MAX_FEATHERS = 24          # feathers per judge run (~10-20 s each on the VLM server, cached afterwards)
VERDICTS = "../data/feather_judge_verdicts.csv"   # your own calls: crop, verdict (good|bad), why

In [ ]:
import base64, html, io, json, sys
from pathlib import Path
sys.path.insert(0, "..")
import pandas as pd
from IPython.display import display, HTML, Markdown
from PIL import Image
from src import prompt_lab as pl

pd.set_option("display.max_colwidth", 120)
def thumb(path, size=(420, 320)):
    im = Image.open(path).convert("RGB"); im.thumbnail(size)
    b = io.BytesIO(); im.save(b, "JPEG", quality=80)
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(b.getvalue()).decode()}" style="max-height:{size[1]}px">'

P = pl.current_prompts()
display(Markdown(f"**SAM 3 prompts**: `{P['sam3_prompts']}` (retry pass adds `{P['sam3_retry_prompts']}`), "
                 f"confidence `{P['sam3_conf']}`"))
display(Markdown("**Mask clean-up**: " + ", ".join(f"`{k}={v}`" for k, v in P["postprocess"].items())))
display(Markdown("**Per-feather judge prompt** (`feather_judge.prompt`):\n\n```text\n" + P["feather_judge_prompt"] + "\n```"))
display(Markdown("**Feather score** (computed from the checks, for sorting only; decisions use the checks): "
                 + ", ".join(f"`{k}={v}`" for k, v in P["feather_score"].items() if k != "_comment")))
display(Markdown("**Missed-feather prompt** (`inventory.prompt`):\n\n```text\n" + P["inventory_prompt"] + "\n```"))

## 2. Feather judge prompt: A vs B
`PROMPT_A` is the current rubric. Edit `PROMPT_B` and run the next cells. Only new (prompt, crop) pairs call the model.
The model answers fixed checks (outline left/right/top/bottom complete|cut, holes or gaps, one feather, fragment,
other feather / other material inside); `score` is derived from them (10 = all passed; -2 per cut side, -3 holes,
-4 not one feather or fragment, -3 other feather, -2 other material), `model_score` is the model's own number.
Holes and split masks are also measured on the mask (`holes`, `mask_parts`) and override a clean answer.

In [ ]:
PROMPT_A = P["feather_judge_prompt"]
PROMPT_B = PROMPT_A   # edit a copy here, e.g. PROMPT_A.replace("old wording", "new wording")
print(PROMPT_B)

In [ ]:
feathers = pl.list_feathers(SEGMENTATION_DIR, SLIDES, MAX_FEATHERS)
print(len(feathers), "feathers from", len({f["file_id"] for f in feathers}), "slides")
A = pl.run_judge(PROMPT_A, feathers)
B = pl.run_judge(PROMPT_B, feathers)
table, stats = pl.compare(A, B, VERDICTS)
if stats is not None:
    display(Markdown("**Agreement with your good/bad calls** (prompt says good = the feather passed every check)"))
    display(stats)
per_check = {side: pl.check_agreement(df, VERDICTS) for side, df in (("A", A), ("B", B))}
if len(per_check["A"]):
    display(Markdown("**Per check, against your hand checks** (`missed problem` = you flagged it, the model passed it)"))
    display(per_check["A"].merge(per_check["B"], on="check", suffixes=(" A", " B")))
display(Markdown(f"**Scores** A mean {A.score.mean():.2f}, B mean {B.score.mean():.2f}; "
                 f"{(table.delta.abs() >= 2).sum()} feathers differ by 2 or more"))
display(table.sort_values("delta", key=abs, ascending=False))

### Where A and B disagree most (or the lowest scores)
Each panel is what the model saw: slide around the feather with the mask outline (left), saved crop (right).

In [ ]:
N_SHOW = 8
show = table.assign(k=table.delta.abs().fillna(0) * 10 - table[["score_A", "score_B"]].min(axis=1).fillna(0))
show = show.sort_values("k", ascending=False).head(N_SHOW)
panels = {r.crop: r.panel for r in A.itertuples() if isinstance(getattr(r, "panel", None), str)}
cells_html = "".join(
    f'<div style="display:inline-block;width:440px;margin:6px;vertical-align:top;font:11px sans-serif">'
    f'{thumb(panels[r.crop]) if r.crop in panels else ""}<br><b>{html.escape(r.file_id)} {r.seq}</b><br>'
    f'A {r.score_A}: {html.escape(str(r.notes_A))}<br>B {r.score_B}: {html.escape(str(r.notes_B))}</div>'
    for r in show.itertuples())
display(HTML(cells_html))

### Your verdicts (optional, makes the comparison objective)
Run the next cell once: it writes `data/feather_judge_verdicts.csv` listing these crops. Open it in JupyterLab and, per crop, put `good` or `bad` in `verdict` and, for the per-check calibration, your answer to each check: `outline_left/right/top/bottom` = `cut` or `complete`; `holes_or_gaps`, `fragment`, `single_feather`, `other_feather_inside`, `other_material_inside` = `true` or `false` (blank = not checked). Save and re-run the A/B cell. Existing calls are kept.

In [ ]:
print(pl.write_verdict_template(feathers, VERDICTS))

### Keep the better prompt
Set `SAVE = "B"` and run: it writes `PROMPT_B` into `configs/segmentation.json`, which the pipeline uses from the next slide on (commit it to ship it to the workers).

In [ ]:
SAVE = None   # "B" to save PROMPT_B as the feather judge prompt
if SAVE == "B":
    pl.save_prompt("feather_judge", PROMPT_B); print("saved")

## 2b. Missed-feather prompt (inventory)
The slide check numbers every segmented feather on the overlay and asks the model for the total and any feather
without an outline. To test it with a known answer, these cells rebuild each slide's masks from its crops and **hide one
feather** (`HIDE`, 0 = leftmost): a correct answer names exactly that letter as missed. `HIDE = None` shows all masks
(correct answer: nothing missed). Edit `INV_PROMPT_B` to try a variant.

In [ ]:
INV_SLIDES = ["A0012 1996-im1058.jpg", "BX40 398 2003-im1818.jpg", "A1340 2005-im1313.jpg", "BD31940 1996-im0912.jpg",
              "AR36456 1993-im0062.jpg", "BZ10105 2003-im1989.jpg"]
HIDE = 1                                   # 0-based position of the feather to hide; None = hide nothing
INV_PROMPT_A = pl.load_seg_config()["inventory"]["prompt"]
INV_PROMPT_B = INV_PROMPT_A                # edit a copy here
ia = pl.run_inventory(INV_PROMPT_A, SEGMENTATION_DIR, INV_SLIDES, HIDE)
ib = pl.run_inventory(INV_PROMPT_B, SEGMENTATION_DIR, INV_SLIDES, HIDE) if INV_PROMPT_B != INV_PROMPT_A else ia
display(Markdown(f"**A correct {int(ia.correct.sum())}/{len(ia)}, B correct {int(ib.correct.sum())}/{len(ib)}**"))
display(ia.drop(columns="overlay").merge(ib[["file_id", "found_missed", "correct", "inventory_status"]], on="file_id",
                                         suffixes=("_A", "_B")))
display(HTML("".join(f'<div style="display:inline-block;margin:4px">{thumb(r.overlay, (380, 420))}<br>'
                     f'{html.escape(r.file_id)}: hidden {r.hidden}, found {r.found_missed or "none"}</div>'
                     for r in ia.itertuples())))


In [ ]:
SAVE_INV = None   # "B" to save INV_PROMPT_B as the inventory prompt
if SAVE_INV == "B":
    pl.save_prompt("inventory", INV_PROMPT_B); print("saved")


## 3. SAM 3 prompts and settings (on the workers)
Each experiment sends the slides through the workers' normal pipeline with `options={'segmentation': TRY}` and copies
the results back (cached by slides + settings). The workers must run this branch's code: the first cell checks.

In [ ]:
ok, registered = pl.workers_have_options_support()
print("workers run the new code:", ok)
if not ok:
    print("Deploy first, e.g. FEATHER_REPO_BRANCH=feather-judge ./run_cluster.sh (from a machine with node "
          "access), or merge the branch. Registered tasks now:", registered)

In [ ]:
SEG_SLIDES = ["A3818 2002-im1708.jpg", "A0012 1996-im1058.jpg", "BX40 398 2003-im1818.jpg", "A2218 1998-im1452.jpg",
              "BB18346 1994-im0622.jpg", "BZ11744 2008-im2397.jpg"]
TRY = {                                   # anything from configs/segmentation.json; missing keys keep the defaults
    "sam3": {"prompts": ["bird feather"], "conf": 0.25},
    "postprocess": {"enabled": True},
}
RUN_SAM3 = False                          # True to send the slides (baseline once, then TRY)
if RUN_SAM3 and ok:
    base_dir = pl.run_seg_experiment(SEG_SLIDES, None, "baseline")
    try_dir = pl.run_seg_experiment(SEG_SLIDES, TRY, "try")
    base, tri = pl.seg_summary(base_dir), pl.seg_summary(try_dir)
    both = base.drop(columns="overlay").merge(tri.drop(columns="overlay"), on=["file_id", "hand_count"],
                                               suffixes=("_base", "_try"))
    display(both)
    for s in SEG_SLIDES:
        b, t = base.set_index("file_id").overlay.get(s), tri.set_index("file_id").overlay.get(s)
        display(HTML(f"<b>{s}</b><br>baseline {thumb(b, (480, 520)) if b else ''} try {thumb(t, (480, 520)) if t else ''}"))

### What the SAM 3 knobs do
- `sam3.prompts`: the concept text SAM 3 segments, e.g. `"bird feather"`, `"feather"`, `"single feather"`; several prompts are unioned, then de-duplicated.
- `sam3.conf`: minimum instance confidence; higher drops faint slivers but can lose pale feathers.
- `sam3.nms_iou`: box overlap above which the weaker of two instances is dropped.
- `postprocess.fragment_area_frac` / `fragment_width_frac`: how small/narrow (vs the slide's typical feather) a mask must be to count as a fragment and be merged into a feather it touches.
- `postprocess.contained_frac`: share of a mask inside another for it to be absorbed.
- `postprocess.align_*`: when two pieces on one axis count as one feather cut by a board.
- `crop.max_feathers`, `crop.trim_bottom_frac`: crops kept per slide, bottom trim (glue/quill base).